# 01 - Data Exploration

Explore the Group 5 ACORN segments, historical consumption, weather relationships, and temporal patterns.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from group5_energy.config import ACORN_GROUPS
from group5_energy.pipeline import load_daily_history, load_half_hourly_history, load_daily_weather

sns.set_theme(style="whitegrid")

In [ ]:
daily = load_daily_history()
half = load_half_hourly_history()
weather = load_daily_weather()

print("Daily rows:", daily.shape)
print("Half-hourly rows:", half.shape)
print("ACORN groups:", ACORN_GROUPS)
print("Daily date range:", daily["Date"].min(), "to", daily["Date"].max())
print("Half-hourly date range:", half["DateTime"].min(), "to", half["DateTime"].max())

In [ ]:
daily.groupby("Acorn")["Conso_kWh"].agg(["count", "mean", "min", "max"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for acorn, group in daily.sort_values("Date").groupby("Acorn"):
    rolling = group.set_index("Date")["Conso_kWh"].rolling(14, min_periods=1).mean()
    ax.plot(rolling.index, rolling.values, label=f"{acorn} - {ACORN_GROUPS[acorn]}")
ax.set_title("Daily electricity consumption, 14-day rolling mean")
ax.set_ylabel("kWh")
ax.legend()
plt.show()

In [ ]:
profile = half.copy()
profile["half_hour_slot"] = profile["DateTime"].dt.hour * 2 + (profile["DateTime"].dt.minute // 30)
profile = profile.groupby(["Acorn", "half_hour_slot"], as_index=False)["Conso_moy"].mean()
profile["time_of_day"] = profile["half_hour_slot"] / 2

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=profile, x="time_of_day", y="Conso_moy", hue="Acorn", ax=ax)
ax.set_title("Typical half-hourly profile")
ax.set_xlabel("Hour of day")
plt.show()

In [ ]:
daily_weather = daily.merge(weather[["Date", "temperatureMean"]], on="Date", how="left")
daily_weather.groupby("Acorn").apply(
    lambda group: group["Conso_kWh"].corr(group["temperatureMean"]),
    include_groups=False,
).rename("temperature_consumption_corr").round(3)